# 20: Real-world photos and in-the-wild generators (E10)

E9 reached a ROC-AUC of 0.999 on OpenFake, yet it called our own iPhone photograph AI-generated. All real images in OpenFake come from two collections, DOCCI and ImageNet, so the detector learned what *those* photographs look like. A phone photo, with its own sharpening and noise reduction, was outside that range.

E10 adds four public sources that OpenFake does not cover:

| Source | What | Licence |
|---|---|---|
| Wikimedia Commons | real photos at original resolution from 57 recent phone models (2019 to 2026): Samsung, Google Pixel, Xiaomi/Redmi/Poco, OnePlus, Vivo, Oppo, Realme, Motorola, Nothing, Honor, Huawei, Tecno, Infinix and iPhone, 30 per model, chosen by the "Taken with ..." categories that Commons fills from the EXIF data | free licences (CC BY, CC BY-SA, public domain) |
| Unsplash Lite | real photographs at original resolution; all phone photos (camera make in EXIF: Apple, Samsung, Google, ...) and 1,500 photos from other cameras | Unsplash dataset terms (research use; images not redistributed, only features kept) |
| ShareGPT-4o-Image | images generated by ChatGPT (GPT-4o) | Apache 2.0 |
| bitmind/nano-banana | images generated by Gemini 2.5 Flash Image (Nano Banana) | MIT |

Each source is split 70 / 15 / 15 into training, calibration and test images. Commons and Unsplash are split by photographer, so the test photos come from photographers never seen in training. Results are reported separately for Android phones and iPhones, and per phone brand. The new data is weighted to make up 30% of the training signal, because it is small next to the 445,000 OpenFake tiles.

E8, E9 and E10 are then compared on the held-out phone photos, the new generators, OpenFake test files 0 to 2 and the E6 WhatsApp pairs. Features of every test image are saved on Drive, so later detectors can be compared without downloading again.

GPU runtime needed. The first run takes about 2 hours; every stage is saved to Drive, so a restart resumes. The last section uploads the detector and needs a Hugging Face **write** token.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub pillow-heif

import hashlib
import io
import itertools
import json
import os
import tarfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pillow_heif
import pyarrow.parquet as pq
import requests
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, roc_curve
from transformers import CLIPVisionModelWithProjection

pillow_heif.register_heif_opener()
Image.MAX_IMAGE_PIXELS = None
ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
E8_DATA = ROOT / "processed_openfake_train_full"
DEPLOY = ROOT / "processed_deploy"
DOWNLOADS = Path("/content/e10_downloads")
DOWNLOADS.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
USER_AGENT = {"User-Agent": "ai-image-forensics-student-project (https://github.com/Tanish-237/ai-image-forensics)"}
COMMONS_API = "https://commons.wikimedia.org/w/api.php"
PER_PHONE_MODEL = 30
ANDROID_MODELS = [
    "Samsung Galaxy S25 Ultra", "Samsung Galaxy S24 Ultra", "Samsung Galaxy S24", "Samsung Galaxy S23 Ultra", "Samsung Galaxy S23",
    "Samsung Galaxy S22 5G", "Samsung Galaxy S21 5G", "Samsung Galaxy A57 5G", "Samsung Galaxy A54 5G", "Samsung Galaxy A35 5G",
    "Samsung Galaxy A16", "Samsung Galaxy A14 5G", "Samsung Galaxy M31",
    "Google Pixel 10 Pro", "Google Pixel 9 Pro", "Google Pixel 9", "Google Pixel 8 Pro", "Google Pixel 8a", "Google Pixel 7", "Google Pixel 6",
    "Xiaomi 15 Ultra", "Xiaomi 14T Pro", "Xiaomi Redmi Note 14 Pro+", "Xiaomi Redmi Note 13 Pro", "Xiaomi Redmi Note 12 Pro+",
    "Xiaomi Redmi 10A", "Xiaomi Poco X3 Pro",
    "OnePlus 11 5G", "OnePlus Nord CE 3 Lite 5G", "OnePlus Nord 2T 5G", "OnePlus Nord",
    "Vivo X200 Pro", "Vivo V40", "Vivo Y50", "Oppo Reno8 Lite 5G", "Realme 10", "Realme 9 Pro+", "Realme 8 Pro",
    "Motorola Edge 60 Pro", "Motorola Edge 40 Neo", "Nothing Phone (2a)", "Nothing Phone (1)",
    "Honor Magic 6 Pro", "Huawei Nova 11", "Huawei P30 Pro", "Tecno Camon 19", "Infinix Smart 9",
]
IPHONE_MODELS = ["iPhone 17 Pro", "iPhone 16 Pro", "iPhone 15 Pro Max (main camera)", "iPhone 14 Pro", "iPhone 14", "iPhone 13",
                 "iPhone 13 Pro Max", "iPhone 12 Pro", "iPhone 11", "iPhone SE (3rd generation)"]
UNSPLASH_ZIP = "https://unsplash-datasets.s3.amazonaws.com/lite/latest/unsplash-research-dataset-lite-latest.zip"
PHONE_MAKES = ("apple", "samsung", "google", "xiaomi", "oneplus", "huawei", "oppo", "vivo", "motorola", "realme",
               "honor", "lge", "lg electronics", "nothing", "sony mobile")
OTHER_CAMERAS = 1500
GPT4O_URL = "https://huggingface.co/datasets/FreedomIntelligence/ShareGPT-4o-Image/resolve/main/text_to_image_part_0.tar"
GPT4O_IMAGES = 2000
NANO_IMAGES = 2000
NEW_DATA_WEIGHT = 0.3            # share of the training signal given to the new sources
TRAIN_FILES = list(range(4, 13))
VAL_FILE = 12
CALIBRATION_FILE = 3
TEST_FILES = [0, 1, 2]
TILES_PER_IMAGE = 4
TEST_PER_SOURCE = 30
TEST_PER_REAL_SOURCE = 70
CALIBRATION_PER_CLASS = 300
SEARCH_TILES = 120_000
WA_LONG_SIDE = 1600
WA_QUALITY = 80
SIZE = 224
MAX_TILES = 9
SEED = 42
WORKERS = 8
REPO = "tanish-237/ai-image-forensics-models"
CLIP_NAME = "openai/clip-vit-large-patch14"


def test_file(i):
    return f"core/test-{i:05d}-of-00013.parquet"


def split_of(group):
    """Deterministic 70/15/15 split by a group key (photographer or image name)."""
    h = int(hashlib.md5(group.encode()).hexdigest(), 16) % 100
    return "train" if h < 70 else "calibration" if h < 85 else "test"

## Helper functions

In [ ]:
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


def jpeg_roundtrip(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=int(quality))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def share_like_whatsapp(img):
    """Simulated WhatsApp share: long side at most 1600 pixels, then JPEG quality 80."""
    w, h = img.size
    scale = min(1.0, WA_LONG_SIDE / max(w, h))
    if scale < 1:
        img = img.resize((round(w * scale), round(h * scale)), Image.BICUBIC)
    return jpeg_roundtrip(img, WA_QUALITY)


def tile_positions(width, height, size=SIZE, max_tiles=MAX_TILES):
    """Evenly spaced, 16-pixel-aligned tile corners covering the whole image (same grid as the website)."""
    per_side = int(np.sqrt(max_tiles))
    xs = np.unique(np.linspace(0, width - size, min(per_side, max(1, width // size))).astype(int) // 16 * 16)
    ys = np.unique(np.linspace(0, height - size, min(per_side, max(1, height // size))).astype(int) // 16 * 16)
    return [(x, y) for y in ys for x in xs]


def enlarge_if_small(img):
    if min(img.size) >= SIZE:
        return img
    factor = SIZE / min(img.size)
    return img.resize((max(SIZE, round(img.size[0] * factor)), max(SIZE, round(img.size[1] * factor))), Image.BICUBIC)


def tiles_of(img, positions=None):
    img = enlarge_if_small(img)
    positions = tile_positions(*img.size) if positions is None else positions
    return np.stack([np.asarray(img.crop((x, y, x + SIZE, y + SIZE))) for x, y in positions])

## CLIP feature extractor (frozen)

In [ ]:
dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def clip_features(arr):
    out = []
    for s in range(0, len(arr), 128):
        x = torch.from_numpy(np.array(arr[s:s + 128])).to(device).permute(0, 3, 1, 2).float() / 255
        if x.shape[-1] != CLIP_SIZE:
            x = F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        out.append(F.normalize(emb, dim=1).cpu().numpy())
    return np.concatenate(out)


def as_distributed(img, rng):
    """Bias control as in notebook 03: a non-JPEG image gets one JPEG compression at a quality drawn from the real images."""
    rgb = img.convert("RGB")
    return rgb if img.format == "JPEG" else jpeg_roundtrip(rgb, rng.choice(REAL_QUALITIES))


e8_meta = pd.read_csv(E8_DATA / "meta.csv")
REAL_QUALITIES = e8_meta.loc[(e8_meta["label"] == "real") & (e8_meta["split"] == "train"), "jpeg_quality"].dropna().astype(int).to_numpy()
print(f"{len(REAL_QUALITIES)} real JPEG qualities, median {np.median(REAL_QUALITIES):.0f}")

## Feature extraction

Every image is used in two versions, as distributed (for training, non-JPEG images get one JPEG compression at a quality drawn from the real images, as in notebook 03) and after a simulated WhatsApp share. Training images contribute four random tiles of the website's grid per version; calibration and test images contribute the full grid, as on the website.

In [ ]:
def download(url, tries=3):
    for attempt in range(tries):
        try:
            r = requests.get(url, headers=USER_AGENT, timeout=120)
            r.raise_for_status()
            return r.content
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(10 * (attempt + 1))


def prepare(item, split=None):
    """Opens one image and returns the tiles of both versions, or None if it cannot be used."""
    try:
        data = item["bytes"] if "bytes" in item else download(item["url"])
        img = Image.open(io.BytesIO(data))
        img.load()
    except Exception:
        return None
    if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE:
        return None
    split = split or split_of(item["group"])
    rng = np.random.default_rng(int(hashlib.md5(item["key"].encode()).hexdigest()[:8], 16))
    fmt = img.format
    rgb = ImageOps.exif_transpose(img).convert("RGB")
    # bias control only for training; calibration and test images are scored exactly as the website sees them
    original = rgb if fmt == "JPEG" or split != "train" else jpeg_roundtrip(rgb, rng.choice(REAL_QUALITIES))
    views = []
    for version, view in [("original", original), ("shared", share_like_whatsapp(rgb))]:
        view = enlarge_if_small(view)
        grid = tile_positions(*view.size)
        if split == "train":
            grid = [grid[k] for k in rng.choice(len(grid), size=min(TILES_PER_IMAGE, len(grid)), replace=False)]
        views.append((version, tiles_of(view, grid)))
    return item, split, views


def build(name, items, split=None):
    """Features of all usable images from `items`, saved as e10_<name>.npz (features) and .csv (one row per tile)."""
    out = DEPLOY / f"e10_{name}"
    if out.with_suffix(".csv").exists():
        print(f"{name}: features already on Drive")
    else:
        start, feats, rows, image_id = time.time(), [], [], 0
        items = iter(items)
        with ThreadPoolExecutor(WORKERS) as pool:
            while chunk := list(itertools.islice(items, 64)):
                tiles = []
                for result in pool.map(lambda it: prepare(it, split), chunk):
                    if result is None:
                        continue
                    item, image_split, views = result
                    for version, t in views:
                        tiles.append(t)
                        rows += [{"image": image_id, "key": item["key"], "source": item["source"], "detail": item.get("detail", ""),
                                  "y": item["y"], "split": image_split, "version": version}] * len(t)
                    image_id += 1
                if tiles:
                    feats.append(clip_features(np.concatenate(tiles)))
        np.save(out.with_suffix(".npy"), np.concatenate(feats).astype(np.float16))
        pd.DataFrame(rows).to_csv(out.with_suffix(".csv"), index=False)
        print(f"{name}: {image_id} images in {time.time() - start:.0f}s")
    meta = pd.read_csv(out.with_suffix(".csv"), keep_default_na=False)
    return np.load(out.with_suffix(".npy")), meta

## New sources: phone photos from Commons and Unsplash, ChatGPT (GPT-4o) and Gemini (Nano Banana) images

In [ ]:
def commons_files(model):
    """JPEG files of at least 3 megapixels in the Commons category 'Taken with <model>'."""
    params = {"action": "query", "generator": "categorymembers", "gcmtitle": f"Category:Taken with {model}", "gcmtype": "file",
              "gcmlimit": "500", "prop": "imageinfo", "iiprop": "url|size|mime|user", "format": "json"}
    pages = {}
    while len(pages) < 1000:
        d = requests.get(COMMONS_API, params=params, headers=USER_AGENT, timeout=60).json()
        for page in d.get("query", {}).get("pages", {}).values():
            if "imageinfo" in page:
                pages[page["pageid"]] = page
        if "continue" not in d:
            break
        params.update(d["continue"])
    files = [(pid, p["imageinfo"][0]) for pid, p in sorted(pages.items())]
    return [(pid, i) for pid, i in files if i["mime"] == "image/jpeg" and i["width"] * i["height"] >= 3_000_000]


def commons_items(models, source):
    rng = np.random.default_rng(SEED)
    for model in models:
        files = commons_files(model)
        for k in rng.choice(len(files), size=min(PER_PHONE_MODEL, len(files)), replace=False):
            pid, info = files[k]
            yield {"key": f"commons-{pid}", "group": f"commons-{info['user']}", "source": source, "detail": model, "y": 0, "url": info["url"]}


commons = [build("commons_android", commons_items(ANDROID_MODELS, "Commons Android")),
           build("commons_iphone", commons_items(IPHONE_MODELS, "Commons iPhone"))]

zip_path = DOWNLOADS / "unsplash_lite.zip"
if not zip_path.exists():
    zip_path.write_bytes(download(UNSPLASH_ZIP))
with zipfile.ZipFile(zip_path) as z:
    name = next(n for n in z.namelist() if n.startswith("photos.tsv"))
    photos = pd.read_csv(z.open(name), sep="\t", keep_default_na=False, quoting=3, dtype=str)
make = photos["exif_camera_make"].str.strip().str.lower()
model = photos["exif_camera_model"].str.lower()
# Samsung also made cameras (NX, Galaxy Camera EK-...), which are not phones
is_phone = (make.str.startswith(PHONE_MAKES) & (~make.str.startswith("apple") | model.str.contains("iphone"))
            & ~(make.str.startswith("samsung") & model.str.match(r"(nx|ek-|wb|es|st|pl)")))
phones = photos[is_phone].assign(source="Unsplash phone")
others = photos[~is_phone & (make != "") & (pd.to_numeric(photos["photo_width"], errors="coerce").fillna(0) >= 1500)]
others = others.sample(n=min(OTHER_CAMERAS, len(others)), random_state=SEED).assign(source="Unsplash camera")
print(f"Unsplash: {len(phones)} phone photos, {len(others)} other camera photos")
display(phones["exif_camera_model"].value_counts().head(10).to_frame("phone photos"))


def unsplash_items(frame):
    for r in frame.itertuples():
        yield {"key": r.photo_id, "group": r.photographer_username, "source": r.source, "detail": r.exif_camera_model,
               "y": 0, "url": r.photo_image_url}


def gpt4o_items():
    with requests.get(GPT4O_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with tarfile.open(fileobj=r.raw, mode="r|") as tar:
            n = 0
            for member in tar:
                if member.isfile() and member.name.endswith(".png"):
                    yield {"key": member.name, "group": member.name, "source": "ChatGPT (GPT-4o)", "y": 1,
                           "bytes": tar.extractfile(member).read()}
                    n += 1
                    if n >= GPT4O_IMAGES:
                        return


def nano_items():
    n = 0
    for k in range(31):
        path = hf_hub_download("bitmind/nano-banana", f"data/train-{k:05d}-of-00031.parquet", repo_type="dataset", local_dir=DOWNLOADS)
        pf = pq.ParquetFile(path)
        for g in range(pf.num_row_groups):
            for item in pf.read_row_group(g, columns=["id", "image"]).to_pylist():
                yield {"key": f"nano-{item['id']}", "group": f"nano-{item['id']}", "source": "Gemini (Nano Banana)", "y": 1,
                       "bytes": item["image"]["bytes"]}
                n += 1
                if n >= NANO_IMAGES:
                    os.remove(path)
                    return
        os.remove(path)


new_data = commons + [build("unsplash_phone", unsplash_items(phones)), build("unsplash_camera", unsplash_items(others)),
            build("gpt4o", gpt4o_items()), build("nano_banana", nano_items())]
new_x = np.concatenate([x for x, _ in new_data])
new_meta = pd.concat([m for _, m in new_data], ignore_index=True)
new_meta["image"] = new_meta["source"] + "-" + new_meta["image"].astype(str)
display(new_meta.drop_duplicates("image").groupby(["source", "split"]).size().unstack())

## Fit E10

The same classifier as E9 (logistic regression on CLIP features, balanced class weights) on the E9 training data plus the new training images. The regularisation strength is chosen on the E9 validation tiles plus the new calibration images.

In [ ]:
e8_meta = pd.read_csv(E8_DATA / "meta.csv")
e8_y = (e8_meta["label"] == "fake").astype(int).to_numpy()
e8_train = (e8_meta["split"] == "train").to_numpy()
parts = {i: np.load(DEPLOY / f"train_{i:02d}.npz") for i in TRAIN_FILES}
new_train = (new_meta["split"] == "train").to_numpy()
new_cal = (new_meta["split"] == "calibration").to_numpy()

blocks = [(np.load(E8_DATA / "clip_train.npy"), e8_y[e8_train], False, False),
          (np.load(E8_DATA / "clip_val.npy"), e8_y[~e8_train], True, False)]
blocks += [(parts[i]["features"], parts[i]["y"], i == VAL_FILE, False) for i in TRAIN_FILES]
blocks += [(new_x[new_train], new_meta["y"].to_numpy()[new_train], False, True),
           (new_x[new_cal], new_meta["y"].to_numpy()[new_cal], True, True)]
n = sum(len(b[1]) for b in blocks)
all_x, all_y = np.empty((n, new_x.shape[1])), np.empty(n, dtype=int)
is_val, is_new = np.empty(n, dtype=bool), np.empty(n, dtype=bool)
row = 0
for x, y, val, new in blocks:
    all_x[row:row + len(y)], all_y[row:row + len(y)] = x, y
    is_val[row:row + len(y)], is_new[row:row + len(y)] = val, new
    row += len(y)
del blocks, parts
fit = ~is_val
weight = np.ones(n)
weight[is_new] = NEW_DATA_WEIGHT / (1 - NEW_DATA_WEIGHT) * (fit & ~is_new).sum() / (fit & is_new).sum()
print(f"{n} tiles, {(fit & is_new).sum()} new training tiles weighted x{weight[is_new][0]:.1f}")

e10_path = DEPLOY / "clip_lr_e10.npz"
if e10_path.exists():
    saved = np.load(e10_path)
    e10_w, e10_b, C = saved["coef"].reshape(-1), float(saved["intercept"]), float(saved["C"])
    print(f"E10 loaded from Drive (C = {C})")
else:
    rng = np.random.default_rng(SEED)
    search_rows = rng.choice(np.flatnonzero(fit), size=min(SEARCH_TILES, fit.sum()), replace=False)
    val_rows = np.flatnonzero(is_val)
    search = []
    for C in [1, 10, 100]:
        c = LogisticRegression(C=C, max_iter=2000, class_weight="balanced").fit(all_x[search_rows], all_y[search_rows], sample_weight=weight[search_rows])
        p = c.predict_proba(all_x[val_rows])[:, 1]
        search.append((float(log_loss(all_y[val_rows], p, sample_weight=weight[val_rows])), C))
        print(f"C={C}: weighted validation loss {search[-1][0]:.4f}, AUC {roc_auc_score(all_y[val_rows], p):.4f}")
    C = min(search)[1]
    start = time.time()
    model = LogisticRegression(C=C, max_iter=2000, class_weight="balanced").fit(all_x, all_y, sample_weight=weight)
    e10_w, e10_b = model.coef_[0], float(model.intercept_[0])
    np.savez(e10_path, coef=e10_w.astype(np.float32), intercept=np.float32(e10_b), C=C)
    print(f"E10: C = {C}, fitted on {n} tiles in {time.time() - start:.0f}s, saved to Drive")
del all_x

config = json.loads(Path(hf_hub_download(REPO, "config.json")).read_text())
e8 = np.load(hf_hub_download(REPO, "m2_lr.npz"))
e9 = np.load(hf_hub_download(REPO, "clip_lr_deploy.npz"))
detectors = {
    "E8": (e8["coef"].reshape(-1), float(e8["intercept"])),
    "E9": (e9["coef"].reshape(-1), float(e9["intercept"])),
    "E10": (e10_w, e10_b),
}
thresholds = {"E8": config["thresholds"]["M2"], "E9": config["detector"]["threshold"]}


def image_scores(x, meta):
    """Website score of every image and version: the mean of its tile probabilities, for every detector."""
    keys = ["image", "version"]
    frame = meta.groupby(keys, sort=False)[[c for c in ["source", "detail", "y", "split"] if c in meta]].first()
    for name, (w, b) in detectors.items():
        p = 1 / (1 + np.exp(-(x.astype(np.float32) @ w.astype(np.float32) + b)))
        frame[name] = pd.Series(p, index=pd.MultiIndex.from_frame(meta[keys])).groupby(level=keys, sort=False).mean()
    return frame.reset_index()

## OpenFake features for calibration and testing (saved, so they are only computed once)

The same images as in notebook 19: 300 real and 300 generated from test file 3, and up to 30 per generator and 70 per real source from each of test files 0 to 2.

In [ ]:
def sample_rows(path, per_label=None, per_source=None, per_real_source=None):
    labels_table = pq.ParquetFile(path).read(columns=["label", "model"]).to_pandas()
    rng = np.random.default_rng(SEED)
    if per_label:
        groups = labels_table.groupby("label")
        limit = lambda name, n: min(n, per_label * 2)
    else:
        groups = labels_table.groupby("model")
        limit = lambda name, n: min(n, per_real_source if labels_table.loc[labels_table["model"] == name, "label"].iat[0] == "real" else per_source)
    chosen = set()
    for name, group in groups:
        chosen |= set(rng.choice(group.index.to_numpy(), size=limit(name, len(group)), replace=False).tolist())
    return chosen


def openfake_items(index, pick, cap_per_label=None):
    path = hf_hub_download("ComplexDataLab/OpenFake", test_file(index), repo_type="dataset", local_dir=DOWNLOADS)
    pf = pq.ParquetFile(path)
    chosen = pick(path)
    start, counts = 0, {"real": 0, "fake": 0}
    for g in range(pf.num_row_groups):
        n = pf.metadata.row_group(g).num_rows
        wanted = [r for r in range(start, start + n) if r in chosen]
        if wanted:
            t = pf.read_row_group(g, columns=["image", "label", "model"]).to_pylist()
            for r in wanted:
                item = t[r - start]
                img = Image.open(io.BytesIO(item["image"]["bytes"]))
                if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE:
                    continue
                if cap_per_label and counts[item["label"]] >= cap_per_label:
                    continue
                counts[item["label"]] += 1
                yield {"key": f"openfake-{index}-{r}", "source": item["model"], "y": int(item["label"] == "fake"),
                       "bytes": item["image"]["bytes"]}
        start += n
    os.remove(path)


cal_x, cal_meta = build("openfake_calibration", openfake_items(CALIBRATION_FILE, lambda p: sample_rows(p, per_label=CALIBRATION_PER_CLASS), CALIBRATION_PER_CLASS), split="calibration")
tests = [build(f"openfake_test_{i:02d}", openfake_items(i, lambda p: sample_rows(p, per_source=TEST_PER_SOURCE, per_real_source=TEST_PER_REAL_SOURCE)), split="test")
         for i in TEST_FILES]
for k, (_, m) in enumerate(tests):
    m["image"] = f"file{TEST_FILES[k]}-" + m["image"].astype(str)
of_test_x = np.concatenate([x for x, _ in tests])
of_test_meta = pd.concat([m for _, m in tests], ignore_index=True)

## Calibrate E10

Youden threshold on the OpenFake calibration images plus the new calibration images, both versions. E8 and E9 keep their own thresholds.

In [ ]:
calibration = pd.concat([image_scores(cal_x, cal_meta), image_scores(new_x[new_cal], new_meta[new_cal])], ignore_index=True)
calibration.to_csv(RESULTS / "e10_calibration.csv", index=False)
fpr, tpr, thr = roc_curve(calibration["y"], calibration["E10"])
finite = np.isfinite(thr)
fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
best = np.argmax(tpr - fpr)
thresholds["E10"] = float(np.clip(thr[best], 0.01, 0.99))
for name in detectors:
    print(f"{name}: calibration AUC {roc_auc_score(calibration['y'], calibration[name]):.3f}, threshold {thresholds[name]:.3f}")

## Results

Correct decisions per source: real images called real, generated images called AI, each detector at its own threshold.

In [ ]:
new_test = (new_meta["split"] == "test").to_numpy()
test = pd.concat([image_scores(new_x[new_test], new_meta[new_test]).assign(group=lambda f: f["source"]),
                  image_scores(of_test_x, of_test_meta).assign(group=lambda f: np.where(f["y"] == 1, "OpenFake generated", "OpenFake real"))],
                 ignore_index=True)
test.to_csv(RESULTS / "e10_test_scores.csv", index=False)
for name in detectors:
    test[f"{name} correct"] = (test[name] >= thresholds[name]).astype(int) == test["y"]

by_group = test.groupby(["group", "version"])[[f"{n} correct" for n in detectors]].mean().unstack("version")
by_group["images"] = test[test["version"] == "original"].groupby("group").size()
by_group.to_csv(RESULTS / "e10_by_group.csv")
display(by_group.round(3))

phones = test[test["group"].isin(["Commons Android", "Commons iPhone"])].copy()
phones["brand"] = phones["detail"].str.replace("Xiaomi Redmi", "Xiaomi").str.split().str[0]
by_brand = phones.groupby(["brand", "version"])[[f"{n} correct" for n in detectors]].mean().unstack("version")
by_brand["images"] = phones[phones["version"] == "original"].groupby("brand").size()
by_brand.to_csv(RESULTS / "e10_phone_brands.csv")
print("Real phone photos called real, per brand (test photos):")
display(by_brand.round(3))

openfake_part = test[test["group"].str.startswith("OpenFake")]
per_source = openfake_part.groupby(["source", "version"])[[f"{n} correct" for n in detectors]].mean().unstack("version")
per_source.to_csv(RESULTS / "e10_openfake_per_source.csv")
display(per_source.round(3))

summary = []
for (part, version), frame in [((p, v), f) for p, sub in [("new sources", test[~test["group"].str.startswith("OpenFake")]), ("OpenFake", openfake_part)]
                               for v, f in sub.groupby("version")]:
    for name in detectors:
        decided = frame[name] >= thresholds[name]
        summary.append({"test": part, "version": version, "detector": name, "auc": roc_auc_score(frame["y"], frame[name]),
                        "real_called_real": float((~decided[frame["y"] == 0]).mean()), "ai_detected": float(decided[frame["y"] == 1].mean())})
summary = pd.DataFrame(summary)
summary.to_csv(RESULTS / "e10_summary.csv", index=False)
display(summary.round(3))

## Real WhatsApp images (E6)

In [ ]:
pairs = pd.read_csv(RESULTS / "e6_whatsapp_predictions.csv")
wa_items = [{"key": f"{when}-{f}", "source": when, "y": int(y), "bytes": (folder / f).read_bytes()}
            for when, folder, column in [("before", ROOT / "whatsapp" / "original", "file"), ("after", ROOT / "whatsapp" / "received", "received_file")]
            for f, y in zip(pairs[column], pairs["y"])]
wa_x, wa_meta = build("whatsapp_pairs", wa_items, split="test")
wa = image_scores(wa_x, wa_meta)
wa = wa[wa["version"] == "original"]
rows = []
for when, frame in wa.groupby("source"):
    for name in detectors:
        decided = frame[name] >= thresholds[name]
        rows.append({"when": when, "detector": name, "auc": roc_auc_score(frame["y"], frame[name]),
                     "real_called_real": float((~decided[frame["y"] == 0]).mean()), "ai_detected": float(decided[frame["y"] == 1].mean())})
whatsapp = pd.DataFrame(rows)
whatsapp.to_csv(RESULTS / "e10_whatsapp.csv", index=False)
display(whatsapp.round(3))

## Chart

In [ ]:
groups = [g for g in ["Commons Android", "Commons iPhone", "Unsplash phone", "Unsplash camera", "OpenFake real", "ChatGPT (GPT-4o)", "Gemini (Nano Banana)", "OpenFake generated"]
          if g in by_group.index]
n_real = sum(g in ("Commons Android", "Commons iPhone", "Unsplash phone", "Unsplash camera", "OpenFake real") for g in groups)
colors = {"E8": "#c3cad4", "E9": "#7fa7dd", "E10": "#2a78d6"}
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), sharey=True)
x = np.arange(len(groups))
for ax, version, title in zip(axes, ["original", "shared"], ["As distributed", "After simulated WhatsApp share"]):
    for k, name in enumerate(detectors):
        values = [by_group.loc[g, (f"{name} correct", version)] * 100 for g in groups]
        bars = ax.bar(x + (k - 1) * 0.27, values, 0.25, color=colors[name], label=name)
        ax.bar_label(bars, fmt="%.0f", fontsize=7)
    ax.set_xticks(x, [g.replace(" (", "\n(") for g in groups], fontsize=8)
    ax.axvline(n_real - 0.5, color="#999999", linewidth=1)
    ax.text((n_real - 1) / 2, 104, "real", ha="center", fontsize=9, color="#555555")
    ax.text((n_real + len(groups) - 1) / 2, 104, "generated", ha="center", fontsize=9, color="#555555")
    ax.set_title(title, fontsize=11, pad=18)
    ax.set_ylim(0, 110)
    ax.grid(axis="y", color="#e5e5e5")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Correct decisions (%)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(RESULTS / "e10_real_world.png", dpi=120)
plt.show()

## Upload to Hugging Face

Adds `clip_lr_e10.npz` and points the website's `detector` entry to it. The E8 and E9 files stay in the repository.

In [ ]:
OUT = Path("/content/e10_upload")
OUT.mkdir(exist_ok=True)
np.savez(OUT / "clip_lr_e10.npz", coef=e10_w.reshape(1, -1).astype(np.float32), intercept=np.array(e10_b, dtype=np.float32))
config["detector"] = {
    "file": "clip_lr_e10.npz",
    "threshold": thresholds["E10"],
    "name": "CLIP + linear classifier, trained on OpenFake, phone photos (Android and iPhone), ChatGPT and Gemini images",
}
config["description"] = "E10 detector"
(OUT / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config["detector"], indent=2))

api = HfApi(token=getpass("Hugging Face write token: "))
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model", commit_message="Add E10 detector")
print(f"Uploaded to https://huggingface.co/{REPO}")